In [1]:
from classes.clase_reportes_new import ReportClassNew
rc = ReportClassNew()

In [ ]:
shopi = rc.consolidar_carpeta(ruta_carpeta=r'G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\conciliaciones_ecommerce\shopify', extension='csv')
import pandas as pd
import numpy as np

# --- 1. Identificar órdenes con SKUs NO válidos ---
ordenes_obs = shopi[
    (~shopi['Lineitem sku'].fillna('').str.startswith(('PCN','TNG','B8','KD'))) &
    (shopi['Lineitem sku'].notna())
].groupby('Name').size().index


# --- 2. Crear dataset agregado por orden ---
ordenes = shopi.groupby('Name', as_index=False).agg({
    'Shipping City': 'first',
    'Shipping': 'first',
    'Subtotal': 'first'
})


# --- 3. Aplicar condiciones a nivel de orden ---
condiciones = [
    (ordenes['Subtotal'] >= 200000) & (~ordenes['Name'].isin(ordenes_obs)),
    (ordenes['Shipping City'] == 'Cali') & (ordenes['Name'].isin(ordenes_obs)),
    (ordenes['Shipping City'] == 'Cali') 
    
    
]

opciones = [
    'No deberia pagar envío',
    'Deberia pagar envío',
    'Cali gratis'
]

ordenes['Observaciones'] = np.select(condiciones, opciones, default='Sin observaciones')


# --- 4. Unir resultado al dataset original ---
shopi = shopi.merge(
    ordenes[['Name', 'Observaciones']],
    on='Name',
    how='left'
)



Buscando archivos con extensión '.csv' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\conciliaciones_ecommerce\shopify
  - Archivo 'orders_export_Mar.csv' leído correctamente.
  - Archivo 'orders_export_Feb.csv' leído correctamente.
  - Archivo 'orders_export_Abr4.csv' leído correctamente.
  - Archivo 'orders_export_Jun.csv' leído correctamente.
  - Archivo 'orders_export_May.csv' leído correctamente.
  - Archivo 'orders_export_Ene.csv' leído correctamente.
  - Archivo 'orders_export_Abr1.csv' leído correctamente.
  - Archivo 'orders_export_Abr2.csv' leído correctamente.
  - Archivo 'orders_export_Abr3.csv' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


In [ ]:
shopi['Created at'].str.split(' ')[]

['2026-03-31', '23:26:19', '-0500']

In [14]:
shopi['Created at'] = pd.to_datetime(shopi['Created at'])

ValueError: time data "30/06/2026 23:46" doesn't match format "%Y-%m-%d %H:%M:%S %z", at position 19372. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [15]:

# --- 5. Exportar ---
shopi.to_csv(
    r'D:\Desktop\shopify_data_proc.csv',
    index=False,
    sep=';',
    decimal=',',
    encoding='utf-8-sig'
)